# Part A: (beta-)VAE and VQ-VAE with PixelCNN Prior

CIFAR-10 image generation assignment - Part A.

Covers:
- beta-VAE trained for beta in {1, 2, 4, 10}: PSNR-vs-epoch and FID comparison
- VQ-VAE trained for codebook size K in {512, 256, 128}: reconstruction quality, discrete latents
- PixelCNN prior trained over the VQ-VAE discrete latents, used to sample new latent codes and generate images
- FID / PSNR / codebook-usage comparison across K

**Before running:** in Colab, go to `Runtime > Change runtime type` and select a GPU.

To reduce Colab GPU time, training uses a 15,000-image subset of CIFAR-10 train and a
3,000-image subset of CIFAR-10 test, with reduced epoch counts. This keeps the whole
notebook runnable in roughly 1-1.5 hours on a free-tier T4 GPU while still producing real
(if not fully converged) metrics. Increase `SUBSET_SIZE` / `EPOCHS_*` below for a
more rigorous final run if you have Colab Pro or more GPU time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || print("No GPU detected - go to Runtime > Change runtime type > GPU")

In [ ]:
import os, time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/samples", exist_ok=True)

## 1. Data: CIFAR-10, normalized to [0,1]

In [ ]:
SUBSET_SIZE = 15000       # of 50,000 train images
TEST_SUBSET_SIZE = 3000   # of 10,000 test images
BATCH_SIZE = 128

transform = T.Compose([T.ToTensor()])  # ToTensor already scales to [0,1]

train_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

g = torch.Generator().manual_seed(SEED)
train_idx = torch.randperm(len(train_full), generator=g)[:SUBSET_SIZE]
test_idx = torch.randperm(len(test_full), generator=g)[:TEST_SUBSET_SIZE]

train_set = Subset(train_full, train_idx)
test_set = Subset(test_full, test_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train subset: {len(train_set)} images, Test subset: {len(test_set)} images")

# sanity check
xb, yb = next(iter(train_loader))
print("Batch shape:", xb.shape, "min/max:", xb.min().item(), xb.max().item())
grid = vutils.make_grid(xb[:32], nrow=8)
plt.figure(figsize=(8,4)); plt.axis("off"); plt.title("Sample CIFAR-10 batch")
plt.imshow(grid.permute(1,2,0).numpy()); plt.show()

## 2. Evaluation utilities: PSNR and FID

FID is computed from pool activations of an ImageNet-pretrained InceptionV3, matching the
standard FID recipe (resize to 299x299, ImageNet normalization, 2048-d pool features,
Frechet distance between Gaussian fits to real vs. generated activations).

In [ ]:
def compute_psnr(x, x_hat, max_val=1.0):
    # x, x_hat: (B,C,H,W) in [0,1]
    mse = torch.mean((x - x_hat) ** 2, dim=[1,2,3]).clamp(min=1e-10)
    psnr = 10 * torch.log10((max_val ** 2) / mse)
    return psnr.mean().item()

In [ ]:
from scipy import linalg
import torchvision.models as tvm

class InceptionFeatureExtractor(nn.Module):
    def __init__(self, device):
        super().__init__()
        weights = tvm.Inception_V3_Weights.IMAGENET1K_V1
        net = tvm.inception_v3(weights=weights, aux_logits=True)
        net.fc = nn.Identity()
        net.eval()
        for p in net.parameters():
            p.requires_grad_(False)
        self.net = net.to(device)
        self.device = device
        self.register_buffer_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
        self.register_buffer_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

    @torch.no_grad()
    def features(self, x):
        # x: (B,3,H,W) in [0,1]
        x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        x = (x - self.register_buffer_mean) / self.register_buffer_std
        feat = self.net(x)
        return feat.detach().cpu().numpy()

_fid_extractor = None
def get_fid_extractor():
    global _fid_extractor
    if _fid_extractor is None:
        _fid_extractor = InceptionFeatureExtractor(device)
    return _fid_extractor

@torch.no_grad()
def extract_features(images, batch_size=64):
    # images: tensor (N,3,H,W) in [0,1] on CPU or GPU
    extractor = get_fid_extractor()
    feats = []
    for i in range(0, images.size(0), batch_size):
        batch = images[i:i+batch_size].to(device)
        feats.append(extractor.features(batch))
    return np.concatenate(feats, axis=0)

def frechet_distance(feat_real, feat_fake, eps=1e-6):
    mu1, mu2 = feat_real.mean(0), feat_fake.mean(0)
    sigma1 = np.cov(feat_real, rowvar=False)
    sigma2 = np.cov(feat_fake, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(fid)

def compute_fid(real_images, fake_images):
    feat_real = extract_features(real_images)
    feat_fake = extract_features(fake_images)
    return frechet_distance(feat_real, feat_fake)

# Cache a fixed batch of real test images to compare against (used by every FID call below)
FID_REF_N = 2000
_real_ref_imgs = torch.cat([b for b, _ in test_loader], dim=0)[:FID_REF_N]
print("FID reference set:", _real_ref_imgs.shape)

## 3. Part A.1 - (beta-)VAE

Encoder/decoder conv architecture for 32x32x3 CIFAR images with a Gaussian latent space.
Trained separately for beta in {1, 2, 4, 10}; loss is reconstruction MSE + beta * KL divergence.

In [ ]:
LATENT_DIM = 128

class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),   # 16x16
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),  # 8x8
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),# 4x4
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), # 2x2
        )
        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 2 * 2)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), # 4x4
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),   # 8x8
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),    # 16x16
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),                                  # 32x32
        )

    def forward(self, z):
        h = self.fc(z).view(-1, 256, 2, 2)
        return self.deconv(h)

class BetaVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar

def vae_loss(x, x_hat, mu, logvar, beta):
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta * kld, recon.item(), kld.item()

In [ ]:
BETAS = [1, 2, 4, 10]
EPOCHS_VAE = 15
LR_VAE = 2e-4

vae_models = {}
vae_psnr_history = {}   # beta -> list of test PSNR per epoch

for beta in BETAS:
    print(f"\n=== Training beta-VAE, beta={beta} ===")
    model = BetaVAE().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR_VAE)
    psnr_hist = []

    for epoch in range(1, EPOCHS_VAE + 1):
        model.train()
        running_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, mu, logvar = model(x)
            loss, recon, kld = vae_loss(x, x_hat, mu, logvar, beta)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running_loss += loss.item()

        # eval PSNR on test subset
        model.eval()
        psnr_vals = []
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_hat, mu, logvar = model(x)
                psnr_vals.append(compute_psnr(x, x_hat))
        epoch_psnr = float(np.mean(psnr_vals))
        psnr_hist.append(epoch_psnr)
        print(f"  epoch {epoch:02d}/{EPOCHS_VAE}  train_loss={running_loss/len(train_loader):.2f}  test_PSNR={epoch_psnr:.2f} dB")

    vae_models[beta] = model
    vae_psnr_history[beta] = psnr_hist

### PSNR vs epoch, per beta

In [ ]:
plt.figure(figsize=(7,5))
for beta in BETAS:
    plt.plot(range(1, EPOCHS_VAE + 1), vae_psnr_history[beta], marker="o", label=f"beta={beta}")
plt.xlabel("Epoch"); plt.ylabel("Test PSNR (dB)")
plt.title("beta-VAE: reconstruction PSNR vs epoch")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("outputs/vae_psnr_vs_epoch.png", dpi=150, bbox_inches="tight")
plt.show()

### FID per beta

FID is computed between real test images and this model's reconstructions of those same
images (standard way to score reconstruction-based realism for VAEs).

In [ ]:
vae_fid = {}
for beta in BETAS:
    model = vae_models[beta]
    model.eval()
    recon_imgs = []
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            x_hat, _, _ = model(x)
            recon_imgs.append(x_hat.cpu())
            if sum(t.size(0) for t in recon_imgs) >= FID_REF_N:
                break
    recon_imgs = torch.cat(recon_imgs, dim=0)[:FID_REF_N]
    fid = compute_fid(_real_ref_imgs, recon_imgs)
    vae_fid[beta] = fid
    print(f"beta={beta:>2}  FID={fid:.2f}")

plt.figure(figsize=(6,4))
plt.bar([str(b) for b in BETAS], [vae_fid[b] for b in BETAS], color="steelblue")
plt.xlabel("beta"); plt.ylabel("FID (lower is better)")
plt.title("beta-VAE: FID vs beta")
plt.savefig("outputs/vae_fid_vs_beta.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Qualitative check: reconstructions for beta=1 (needed later for Part C)
model = vae_models[1]
model.eval()
x, _ = next(iter(test_loader))
x = x.to(device)
with torch.no_grad():
    x_hat, _, _ = model(x)

n = 8
fig, axes = plt.subplots(2, n, figsize=(2*n, 4))
for i in range(n):
    axes[0, i].imshow(x[i].cpu().permute(1,2,0)); axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].cpu().permute(1,2,0)); axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original"); axes[1, 0].set_ylabel("Recon (beta=1)")
plt.suptitle("beta-VAE (beta=1): originals (top) vs reconstructions (bottom)")
plt.savefig("outputs/vae_beta1_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Part A.2 - VQ-VAE with PixelCNN prior

Encoder maps 32x32x3 -> an 8x8 grid of D-dim continuous vectors, which are quantized against
a learned codebook of K embeddings (K in {512, 256, 128}). A PixelCNN with masked
convolutions is then trained autoregressively over the resulting 8x8 grid of discrete code
indices, so that new latent grids (and therefore new images) can be sampled from scratch.

In [ ]:
EMBED_DIM = 64      # D
LATENT_HW = 8        # encoder downsamples 32x32 -> 8x8

class VQVAEEncoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),      # 16x16
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),   # 8x8
            nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),  # 8x8
            nn.Conv2d(128, embed_dim, 1, 1, 0),                                        # 8x8, D channels
        )

    def forward(self, x):
        return self.net(x)

class VQVAEDecoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(embed_dim, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),  # 16x16
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),   # 32x32
            nn.Conv2d(32, 3, 3, 1, 1), nn.Sigmoid(),
        )

    def forward(self, z_q):
        return self.net(z_q)

class VectorQuantizer(nn.Module):
    def __init__(self, K, embed_dim=EMBED_DIM):
        super().__init__()
        self.K = K
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(K, embed_dim)
        self.embedding.weight.data.uniform_(-1.0 / K, 1.0 / K)

    def forward(self, z_e):
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).contiguous().view(-1, D)  # (B*H*W, D)
        dist = (flat.pow(2).sum(1, keepdim=True)
                - 2 * flat @ self.embedding.weight.t()
                + self.embedding.weight.pow(2).sum(1))
        indices = dist.argmin(1)                                  # (B*H*W,)
        z_q = self.embedding(indices).view(B, H, W, D).permute(0, 3, 1, 2).contiguous()
        indices = indices.view(B, H, W)
        return z_q, indices

class VQVAE(nn.Module):
    def __init__(self, K, embed_dim=EMBED_DIM, commitment_cost=0.25):
        super().__init__()
        self.encoder = VQVAEEncoder(embed_dim)
        self.vq = VectorQuantizer(K, embed_dim)
        self.decoder = VQVAEDecoder(embed_dim)
        self.commitment_cost = commitment_cost

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, indices = self.vq(z_e)
        z_q_st = z_e + (z_q - z_e).detach()  # straight-through estimator
        x_hat = self.decoder(z_q_st)
        return x_hat, z_e, z_q, indices

    def loss(self, x, x_hat, z_e, z_q):
        recon = F.mse_loss(x_hat, x)
        codebook_loss = F.mse_loss(z_q, z_e.detach())
        commitment_loss = F.mse_loss(z_e, z_q.detach())
        return recon + codebook_loss + self.commitment_cost * commitment_loss, recon.item()

def codebook_stats(indices, K):
    flat = indices.flatten()
    counts = torch.bincount(flat, minlength=K).float()
    probs = counts / counts.sum().clamp(min=1)
    nz = probs[probs > 0]
    entropy = -(nz * nz.log()).sum()
    perplexity = torch.exp(entropy).item()
    active_codes = int((counts > 0).sum().item())
    return perplexity, active_codes

In [ ]:
K_VALUES = [512, 256, 128]
EPOCHS_VQVAE = 15
LR_VQVAE = 2e-4

vqvae_models = {}
vqvae_psnr_history = {}
vqvae_final_stats = {}   # K -> dict(psnr, perplexity, active_codes, fid)

for K in K_VALUES:
    print(f"\n=== Training VQ-VAE, K={K} ===")
    model = VQVAE(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR_VQVAE)
    psnr_hist = []

    for epoch in range(1, EPOCHS_VQVAE + 1):
        model.train()
        running_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, z_e, z_q, indices = model(x)
            loss, recon = model.loss(x, x_hat, z_e, z_q)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running_loss += loss.item()

        model.eval()
        psnr_vals, perp_vals, active_vals = [], [], []
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_hat, z_e, z_q, indices = model(x)
                psnr_vals.append(compute_psnr(x, x_hat))
                perp, active = codebook_stats(indices, K)
                perp_vals.append(perp); active_vals.append(active)
        epoch_psnr = float(np.mean(psnr_vals))
        psnr_hist.append(epoch_psnr)
        print(f"  epoch {epoch:02d}/{EPOCHS_VQVAE}  train_loss={running_loss/len(train_loader):.4f}  "
              f"test_PSNR={epoch_psnr:.2f} dB  perplexity={np.mean(perp_vals):.1f}/{K}")

    vqvae_models[K] = model
    vqvae_psnr_history[K] = psnr_hist
    vqvae_final_stats[K] = {
        "psnr": epoch_psnr,
        "perplexity": float(np.mean(perp_vals)),
        "active_codes": float(np.mean(active_vals)),
    }

### VQ-VAE reconstruction quality vs K

In [ ]:
plt.figure(figsize=(7,5))
for K in K_VALUES:
    plt.plot(range(1, EPOCHS_VQVAE + 1), vqvae_psnr_history[K], marker="o", label=f"K={K}")
plt.xlabel("Epoch"); plt.ylabel("Test PSNR (dB)")
plt.title("VQ-VAE: reconstruction PSNR vs epoch")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("outputs/vqvae_psnr_vs_epoch.png", dpi=150, bbox_inches="tight")
plt.show()

# Reconstruction FID (recon vs real), same protocol as the VAE section
for K in K_VALUES:
    model = vqvae_models[K]
    model.eval()
    recon_imgs = []
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            x_hat, *_ = model(x)
            recon_imgs.append(x_hat.cpu())
            if sum(t.size(0) for t in recon_imgs) >= FID_REF_N:
                break
    recon_imgs = torch.cat(recon_imgs, dim=0)[:FID_REF_N]
    fid = compute_fid(_real_ref_imgs, recon_imgs)
    vqvae_final_stats[K]["recon_fid"] = fid
    print(f"K={K:>3}  reconstruction FID={fid:.2f}")

### Extract discrete latent maps (needed to train the PixelCNN prior)

In [ ]:
@torch.no_grad()
def extract_all_latents(model, loader, K):
    model.eval()
    all_idx = []
    for x, _ in loader:
        x = x.to(device)
        _, _, _, indices = model(x)
        all_idx.append(indices.cpu())
    return torch.cat(all_idx, dim=0)  # (N, H, W) long

vqvae_latents = {}
for K in K_VALUES:
    latents = extract_all_latents(vqvae_models[K], train_loader, K)
    vqvae_latents[K] = latents
    print(f"K={K}: latent tensor shape {tuple(latents.shape)}, dtype {latents.dtype}")

### PixelCNN prior over the discrete latent grid

Masked convolutions (van den Oord et al.) give a tractable autoregressive model over the
8x8 grid of code indices: mask type 'A' for the first layer (no self-connection), type 'B'
for subsequent layers (allows self-connection so features can compound), raster-scan order.

In [ ]:
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert mask_type in ("A", "B")
        self.register_buffer("mask", self.weight.data.clone())
        _, _, kh, kw = self.weight.shape
        yc, xc = kh // 2, kw // 2
        self.mask.fill_(1)
        self.mask[:, :, yc, xc + (1 if mask_type == "B" else 0):] = 0
        self.mask[:, :, yc + 1:, :] = 0

    def forward(self, x):
        self.weight.data *= self.mask
        return super().forward(x)

class LatentPixelCNN(nn.Module):
    def __init__(self, K, hidden=128, n_layers=6, kernel_size=5):
        super().__init__()
        self.K = K
        pad = kernel_size // 2
        self.embed = nn.Embedding(K, hidden)
        layers = [MaskedConv2d("A", hidden, hidden, kernel_size, padding=pad), nn.ReLU(inplace=True)]
        for _ in range(n_layers - 1):
            layers += [MaskedConv2d("B", hidden, hidden, kernel_size, padding=pad), nn.ReLU(inplace=True)]
        self.net = nn.Sequential(*layers)
        self.out = nn.Conv2d(hidden, K, 1)

    def forward(self, x):
        # x: (B,H,W) long code indices
        h = self.embed(x).permute(0, 3, 1, 2)  # (B, hidden, H, W)
        h = self.net(h)
        return self.out(h)  # (B, K, H, W) logits

    @torch.no_grad()
    def sample(self, n, hw, device):
        self.eval()
        grid = torch.zeros(n, hw, hw, dtype=torch.long, device=device)
        for i in range(hw):
            for j in range(hw):
                logits = self.forward(grid)               # (n, K, H, W)
                probs = F.softmax(logits[:, :, i, j], dim=1)
                grid[:, i, j] = torch.multinomial(probs, 1).squeeze(1)
        return grid

In [ ]:
EPOCHS_PIXELCNN = 15
LR_PIXELCNN = 2e-3
PCNN_BATCH = 128

pixelcnn_models = {}

for K in K_VALUES:
    print(f"\n=== Training PixelCNN prior, K={K} ===")
    latents = vqvae_latents[K].to(device)
    n = latents.size(0)
    model = LatentPixelCNN(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR_PIXELCNN)

    for epoch in range(1, EPOCHS_PIXELCNN + 1):
        model.train()
        perm = torch.randperm(n, device=device)
        running_loss = 0.0
        n_batches = 0
        for i in range(0, n, PCNN_BATCH):
            idx = perm[i:i + PCNN_BATCH]
            batch = latents[idx]                     # (B,H,W)
            logits = model(batch)                     # (B,K,H,W)
            loss = F.cross_entropy(logits, batch)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running_loss += loss.item()
            n_batches += 1
        print(f"  epoch {epoch:02d}/{EPOCHS_PIXELCNN}  CE loss={running_loss/n_batches:.4f}")

    pixelcnn_models[K] = model

### Sample new latent codes with PixelCNN and decode into images

In [ ]:
N_SAMPLES = 64
vqvae_generated = {}

for K in K_VALUES:
    print(f"Sampling {N_SAMPLES} new latent grids for K={K} (this loops over all {LATENT_HW*LATENT_HW} grid cells)...")
    grid = pixelcnn_models[K].sample(N_SAMPLES, LATENT_HW, device)   # (N,H,W) long
    with torch.no_grad():
        z_q = vqvae_models[K].vq.embedding(grid).permute(0, 3, 1, 2).contiguous()
        gen_imgs = vqvae_models[K].decoder(z_q).cpu()
    vqvae_generated[K] = gen_imgs

    grid_img = vutils.make_grid(gen_imgs[:32], nrow=8)
    plt.figure(figsize=(8, 4))
    plt.axis("off"); plt.title(f"VQ-VAE + PixelCNN generated samples (K={K})")
    plt.imshow(grid_img.permute(1, 2, 0).numpy())
    plt.savefig(f"outputs/samples/vqvae_pixelcnn_K{K}.png", dpi=150, bbox_inches="tight")
    plt.show()

### Compare FID, PSNR and codebook quality across K

In [ ]:
for K in K_VALUES:
    fid = compute_fid(_real_ref_imgs, vqvae_generated[K])
    vqvae_final_stats[K]["sample_fid"] = fid
    print(f"K={K:>3}  sample FID (PixelCNN-generated vs real)={fid:.2f}")

print()
print(f"{'K':>5} | {'Recon PSNR (dB)':>16} | {'Recon FID':>10} | {'Sample FID':>10} | {'Perplexity':>10} | {'Active codes':>12}")
print("-" * 78)
for K in K_VALUES:
    s = vqvae_final_stats[K]
    print(f"{K:>5} | {s['psnr']:>16.2f} | {s['recon_fid']:>10.2f} | {s['sample_fid']:>10.2f} | "
          f"{s['perplexity']:>10.1f} | {s['active_codes']:>8.0f}/{K}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar([str(k) for k in K_VALUES], [vqvae_final_stats[k]["psnr"] for k in K_VALUES], color="seagreen")
axes[0].set_title("Reconstruction PSNR vs K"); axes[0].set_xlabel("K"); axes[0].set_ylabel("dB")

axes[1].bar([str(k) for k in K_VALUES], [vqvae_final_stats[k]["sample_fid"] for k in K_VALUES], color="indianred")
axes[1].set_title("Sample FID vs K (lower better)"); axes[1].set_xlabel("K")

axes[2].bar([str(k) for k in K_VALUES], [vqvae_final_stats[k]["perplexity"] for k in K_VALUES], color="steelblue")
axes[2].set_title("Codebook perplexity vs K"); axes[2].set_xlabel("K")

plt.tight_layout()
plt.savefig("outputs/vqvae_comparison_across_K.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save artifacts for Part C

Saves reconstruction/sample grids, metrics, and model checkpoints under `outputs/` so they
can be pulled into the Part C comparative-analysis PDF. In Colab, either download the
`outputs/` folder (zip it first) or mount Google Drive and copy it over.

In [ ]:
import json

torch.save({b: vae_models[b].state_dict() for b in BETAS}, "outputs/beta_vae_checkpoints.pt")
torch.save({k: vqvae_models[k].state_dict() for k in K_VALUES}, "outputs/vqvae_checkpoints.pt")
torch.save({k: pixelcnn_models[k].state_dict() for k in K_VALUES}, "outputs/pixelcnn_checkpoints.pt")

metrics = {
    "beta_vae": {str(b): {"psnr_history": vae_psnr_history[b], "fid": vae_fid[b]} for b in BETAS},
    "vqvae": {str(k): vqvae_final_stats[k] for k in K_VALUES},
}
with open("outputs/part_a_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# 100 samples for Part C: VAE (beta=1) reconstructions and VQ-VAE (K=256) generated samples
model = vae_models[1]; model.eval()
imgs = []
with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        x_hat, _, _ = model(x)
        imgs.append(x_hat.cpu())
        if sum(t.size(0) for t in imgs) >= 100:
            break
vae_beta1_100 = torch.cat(imgs, 0)[:100]
vutils.save_image(vae_beta1_100, "outputs/samples/beta_vae_beta1_100_samples.png", nrow=10)

vqvae_k256_100 = vqvae_generated[256][:100] if vqvae_generated[256].size(0) >= 100 else vqvae_generated[256]
vutils.save_image(vqvae_k256_100, "outputs/samples/vqvae_k256_100_samples.png", nrow=10)

print("Saved checkpoints, metrics, and Part C sample grids under outputs/")

!zip -rq outputs_part_a.zip outputs
print("Zipped -> outputs_part_a.zip (download this from the Colab file browser)")

## 6. Export to HTML (deliverable)

Run this in Colab after all cells above have executed, or use
`File > Download > Download .ipynb` then run the same command locally / in another Colab
cell pointed at the downloaded file.

In [ ]:
NOTEBOOK_NAME = "Part_A_VAE_VQVAE.ipynb"  # update if you renamed the file in Colab
!jupyter nbconvert --to html "{NOTEBOOK_NAME}" 2>/dev/null || print("Save the notebook first (Ctrl+S / Cmd+S), then re-run this cell.")